# test_model_full: full 8,760-hour Gurobi sizing model (Gavin's sandbox)
Same idea as `test_model.ipynb` but **no temperature binning**: every hour of the year is its own pair of variables (heat pump heat, gas heat), so Gurobi chooses the heat pump size **and** the hour-by-hour dispatch against real hourly weather and the **hourly** MISO grid carbon file.

**Needs a full Gurobi license** (e.g. your academic license). The pip-bundled restricted license caps models at 2,000 variables/constraints and will raise `Model too large for size-limited license`; this model has ~17,500 variables. Set `SUBSET_DAYS = 14` to smoke-test under the restricted license (14 evenly spaced days, annualized; a smoke test, not a real result).

**License handling:** gurobipy finds your license itself (`gurobi.lic` in your home folder, or the `GRB_LICENSE_FILE` environment variable). Do **not** copy a license file or key into this repo (it is public); `.gitignore` excludes `gurobi.lic`.

Inputs are the same as `test_model.ipynb`: Patrick's heating physics and curves, Sharvi's `68.09 * kBtu/h + 5100` cost, repo price/emission data, Minneapolis only.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import gurobipy as gp
from gurobipy import GRB
from scipy.interpolate import interp1d

SUBSET_DAYS = None   # None = full year (needs a full license). e.g. 14 = 14 evenly spaced days, annualized, for the restricted license.

In [ ]:
# --- Home (Patrick's defaults) ---
AREA_SQFT, R_VAL, SETPOINT_F, Q_INT_BTUH = 3310.0, 5.065, 68.0, 2000.0
UA = AREA_SQFT / R_VAL
# --- Heat pump (Patrick's curves; capacity scaled from nominal as in his parameter search) ---
COP_PTS  = {47.0: 3.60, 17.0: 2.40, 5.0: 1.80, -22.0: 1.00}
CAP_FRAC = {47.0: 1.00, 17.0: 0.80, 5.0: 0.65, -22.0: 0.35}
MIN_KBTUH, MAX_KBTUH = 18, 60
# --- Cost (Sharvi, equation.py) ---
def capital_cost(kbtuh): return 68.09 * kbtuh + 5100
REBATE_USD = 2000
# --- Prices / emissions (data/processed/model_inputs_minneapolis.json) ---
ELEC_USD_KWH, GAS_USD_KWH_FUEL, FURNACE_AFUE = 0.1644, 0.035, 0.95
GAS_KG_KWH_FUEL = 0.181
YEARS = 15
BTU_PER_KWH = 3412.14

## Hourly weather, load and grid carbon

In [ ]:
w = pd.read_csv("data/processed/msp_weather_TMYx.2011-2025.csv")
g = pd.read_csv("data/processed/miso_hourly_grid_carbon_2025.csv", comment="#")
assert len(w) == 8760 and len(g) == 8760
w["t_f"] = w["drybulb_C"] * 9/5 + 32
w["load_btuh"] = np.maximum(0.0, UA * (SETPOINT_F - w["t_f"]) - Q_INT_BTUH)   # house held at setpoint (free cooling), like the capped Patrick copy
w["grid_kg_kwh"] = g["co2_g_per_kWh"].values / 1000          # hourly MISO 2025, local CST; hour i of both files lines up (hour_of_year)
if SUBSET_DAYS:   # evenly spaced days across the year (a smoke test, not a result)
    days = np.linspace(0, 364, SUBSET_DAYS).round().astype(int)
    w = pd.concat([w.iloc[d * 24:(d + 1) * 24] for d in days]).reset_index(drop=True)
SCALE = 8760 / len(w)                                         # annualization factor (1.0 for the full year)

cop_f = interp1d(sorted(COP_PTS), [COP_PTS[k] for k in sorted(COP_PTS)], bounds_error=False, fill_value=(COP_PTS[min(COP_PTS)], COP_PTS[max(COP_PTS)]))
cap_f = interp1d(sorted(CAP_FRAC), [CAP_FRAC[k] for k in sorted(CAP_FRAC)], bounds_error=False, fill_value=(CAP_FRAC[min(CAP_FRAC)], CAP_FRAC[max(CAP_FRAC)]))
w["cop"], w["cap_frac"] = cop_f(w["t_f"]).astype(float), cap_f(w["t_f"]).astype(float)
L, COP, F, GRID = (w[c].values for c in ["load_btuh", "cop", "cap_frac", "grid_kg_kwh"])
n = len(w)
print(f"{n} hours (x{SCALE:.2f}), annual heat need {L.sum()*SCALE/BTU_PER_KWH:,.0f} kWh, peak {L.max()/1000:.1f} kBtu/h, mean grid {GRID.mean()*1000:.0f} g/kWh")

## The Gurobi model (hourly)
Variables: size `s` (kBtu/h), install binary `y`, and for every hour `h_t` (heat pump heat) and `a_t` (gas backup heat), Btu/h.

- `h_t + a_t = L_t`, `h_t <= 1000 f_t s`, `18 y <= s <= 60 y`
- electric kWh `= h_t / (COP_t 3412)`; gas fuel kWh `= a_t / (AFUE 3412)`
- CO2 `= grid_t x electric + 0.181 x gas fuel` (hourly grid intensity, so Gurobi shifts gas into dirty-grid hours)
- minimize `capex(s) - rebate y + YEARS x bill + carbon price x YEARS x CO2`

In [ ]:
def solve(lam_usd_per_t, verbose=False):
    m = gp.Model("sizing_hourly"); m.Params.OutputFlag = 1 if verbose else 0
    sv = m.addMVar(1, lb=0, ub=MAX_KBTUH); y = m.addVar(vtype=GRB.BINARY)
    h = m.addMVar(n, lb=0); a = m.addMVar(n, lb=0)
    m.addConstr(sv[0] >= MIN_KBTUH * y); m.addConstr(sv[0] <= MAX_KBTUH * y)
    m.addConstr(h + a == L)
    m.addConstr(h - (1000 * F)[:, None] @ sv <= 0)
    elec = (h / (COP * BTU_PER_KWH))                      # kWh each hour
    gas = (a / (FURNACE_AFUE * BTU_PER_KWH))
    bill = SCALE * (ELEC_USD_KWH * elec.sum() + GAS_USD_KWH_FUEL * gas.sum())
    co2 = SCALE * ((GRID * elec).sum() + GAS_KG_KWH_FUEL * gas.sum())   # kg
    m.setObjective(68.09 * sv[0] + (5100 - REBATE_USD) * y + YEARS * bill + lam_usd_per_t / 1000 * YEARS * co2, GRB.MINIMIZE)
    m.optimize()
    size = sv[0].X if y.X > 0.5 else 0.0
    hv, av = h.X, a.X
    e, gk = SCALE * (hv / (COP * BTU_PER_KWH)).sum(), SCALE * (av / (FURNACE_AFUE * BTU_PER_KWH)).sum()
    return dict(lambda_usd_per_t=lam_usd_per_t, size_kbtuh=size, size_tons=size / 12,
                upfront_usd=(capital_cost(size) - REBATE_USD) if size else 0.0,
                annual_cost_usd=ELEC_USD_KWH * e + GAS_USD_KWH_FUEL * gk,
                annual_co2_t=SCALE * ((GRID * hv / (COP * BTU_PER_KWH)).sum() + GAS_KG_KWH_FUEL * (av / (FURNACE_AFUE * BTU_PER_KWH)).sum()) / 1000,
                annual_elec_kwh=e, annual_gas_kwh_fuel=gk, hp_share=hv.sum() / L.sum())

gas_kwh = L.sum() * SCALE / (FURNACE_AFUE * BTU_PER_KWH)
gas_only_cost, gas_only_co2 = GAS_USD_KWH_FUEL * gas_kwh, GAS_KG_KWH_FUEL * gas_kwh / 1000
print(f"Keep gas furnace: ${gas_only_cost:,.0f}/yr, {gas_only_co2:.1f} t CO2/yr")

## Sweep the carbon price (same $/tonne grid as model v1)

In [ ]:
LAMS = [0, 10, 25, 50, 75, 100, 150, 200, 300, 400, 500, 750, 1000]
res = pd.DataFrame([solve(l) for l in LAMS])
res["saves_usd_yr"] = gas_only_cost - res["annual_cost_usd"]
res["payback_yr"] = np.where(res.saves_usd_yr > 0, res.upfront_usd / res.saves_usd_yr, np.nan)
res["co2_cut_t_yr"] = gas_only_co2 - res["annual_co2_t"]
res["usd_per_t_avoided"] = np.where(res.co2_cut_t_yr > 0.01, (res.upfront_usd + YEARS * (res.annual_cost_usd - gas_only_cost)) / (YEARS * res.co2_cut_t_yr), np.nan)
res.round(2)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].step(res.lambda_usd_per_t, res.size_tons, where="post", marker="o"); ax[0].set_xlabel("carbon price ($/t CO2)"); ax[0].set_ylabel("chosen size (tons)"); ax[0].set_title("Recommended size vs carbon price")
ax[1].plot(YEARS * res.annual_co2_t, res.upfront_usd + YEARS * res.annual_cost_usd, "o-"); ax[1].plot([YEARS*gas_only_co2], [YEARS*gas_only_cost], "k*", ms=12, label="keep gas")
ax[1].set_xlabel("15-year CO2 (t)"); ax[1].set_ylabel("15-year cost ($)"); ax[1].set_title("Cost vs carbon"); ax[1].legend()
plt.tight_layout(); plt.show()

## Compare with Patrick's rule (heat pump first), by enumeration
With Patrick's dispatch the only decision is size, so no solver is needed: try every size and score it with the same objective.

In [ ]:
def hp_first(size_kbtuh, lam):
    hp = np.minimum(L, 1000 * F * size_kbtuh); aux = L - hp
    e, gk = hp / (COP * BTU_PER_KWH), aux / (FURNACE_AFUE * BTU_PER_KWH)
    cost = SCALE * (ELEC_USD_KWH * e.sum() + GAS_USD_KWH_FUEL * gk.sum()); co2 = SCALE * ((GRID * e).sum() + GAS_KG_KWH_FUEL * gk.sum()) / 1000
    up = (capital_cost(size_kbtuh) - REBATE_USD) if size_kbtuh else 0.0
    return dict(size_kbtuh=size_kbtuh, upfront_usd=up, annual_cost_usd=cost, annual_co2_t=co2,
                annual_elec_kwh=SCALE * e.sum(), annual_gas_kwh_fuel=SCALE * gk.sum(), score=up + YEARS * cost + lam * YEARS * co2)
rows = []
for lam in LAMS:
    opts = [hp_first(0, lam)] + [hp_first(s, lam) for s in np.arange(MIN_KBTUH, MAX_KBTUH + 0.1, 0.5)]
    rows.append(dict(lambda_usd_per_t=lam, **min(opts, key=lambda r: r["score"])))
res_hf = pd.DataFrame(rows).drop(columns="score")
res_hf.round(2)

## Cross-check: Patrick's hourly RC simulation at the chosen sizes

In [ ]:
import numpy as np
import pandas as pd
from scipy.interpolate import interp1d


def run_thermal_hp_sim(
    df_hourly: pd.DataFrame,
    area_sqft: float = 3310.0,
    r_val: float = 5.065,
    setpoint_f: float = 68.0,
    deadband_f: float = 1.0,
    # Standard multi-point COP ratings dict: {temp_f: COP}
    cop_ratings: dict[float, float] | None = None,
    # Standard multi-point capacity ratings dict: {temp_f: max_btu_per_hr}
    capacity_ratings: dict[float, float] | None = None,
    hp_capacity_tons: float = 3.0,  # Used as fallback baseline if capacity_ratings is None
    gas_furnace_eff: float = 0.80,
    elec_rate_per_kwh: float = 0.15,
    gas_rate_per_therm: float = 1.20,
    grid_co2_kg_per_kwh: float = 0.45,
    gas_co2_kg_per_therm: float = 5.30,
) -> tuple[pd.DataFrame, dict]:
    """Runs an hourly 1-Node RC thermal simulation for a heat pump home with gas backup,

    using 1D linear interpolation for cold-climate COP and capacity curves.
    """
    # 1. Provide default NEEP/AHRI benchmark rating curves if none supplied
    if cop_ratings is None:
        # Standard cold-climate inverter profile (47°F, 17°F, 5°F, -22°F)
        cop_ratings = {47.0: 3.60, 17.0: 2.40, 5.0: 1.80, -22.0: 1.00}

    if capacity_ratings is None:
        # Base 3-ton unit (36,000 BTU/hr nominal at 47°F) with cold weather capacity drop
        capacity_ratings = {
            47.0: hp_capacity_tons * 12000.0,
            17.0: hp_capacity_tons * 12000.0 * 0.78,
            5.0: hp_capacity_tons * 12000.0 * 0.60,
            -22.0: hp_capacity_tons * 12000.0 * 0.30,
        }

    # Sort dictionary keys to ensure ascending arrays for interpolation
    temps_cop = np.array(sorted(cop_ratings.keys()))
    vals_cop = np.array([cop_ratings[t] for t in temps_cop])

    temps_cap = np.array(sorted(capacity_ratings.keys()))
    vals_cap = np.array([capacity_ratings[t] for t in temps_cap])

    # 2. Build SciPy linear interpolation functions
    get_cop_interp = interp1d(
        temps_cop,
        vals_cop,
        kind='linear',
        bounds_error=False,
        fill_value=(vals_cop[0], vals_cop[-1]),  # Clip at boundaries
    )

    get_cap_interp = interp1d(
        temps_cap,
        vals_cap,
        kind='linear',
        bounds_error=False,
        fill_value=(vals_cap[0], vals_cap[-1]),  # Clip at boundaries
    )

    # 3. Setup Building & Simulation State
    n_hours = len(df_hourly)
    ua_btu = area_sqft / r_val
    c_btu = area_sqft * 15.0

    t_in = np.zeros(n_hours)
    q_hp_delivered = np.zeros(n_hours)
    q_aux_delivered = np.zeros(n_hours)
    kwh_hp = np.zeros(n_hours)
    therms_aux = np.zeros(n_hours)
    hvac_action = ['OFF'] * n_hours

    t_in[0] = setpoint_f
    dt_hr = 1.0

    # 4. Hourly Simulation Loop
    for t in range(n_hours - 1):
        t_amb = df_hourly['t_amb_f'].iloc[t]
        q_int = df_hourly['q_int_btu'].iloc[t]

        heat_loss = ua_btu * (t_in[t] - t_amb)
        heat_needed = max(0.0, heat_loss - q_int)

        if t_in[t] < (setpoint_f - deadband_f) or heat_needed > 0:
            # Look up interpolated COP and max capacity for current outdoor temp
            cop_current = float(get_cop_interp(t_amb))
            avail_hp_cap = float(get_cap_interp(t_amb))

            # Heat delivered split
            hp_heat = min(heat_needed, avail_hp_cap)
            aux_heat = max(0.0, heat_needed - hp_heat)

            # Energy inputs
            hp_elec_kwh = (hp_heat / max(1.0, cop_current)) / 3412.14
            gas_therms = (aux_heat / gas_furnace_eff) / 100000.0

            if hp_heat > 0 and aux_heat > 0:
                hvac_action[t] = 'HP_AND_GAS_AUX'
            elif hp_heat > 0:
                hvac_action[t] = 'HP_ONLY'
            elif aux_heat > 0:
                hvac_action[t] = 'GAS_AUX_ONLY'
        else:
            hp_heat = 0.0
            aux_heat = 0.0
            hp_elec_kwh = 0.0
            gas_therms = 0.0
            hvac_action[t] = 'OFF'

        q_hp_delivered[t] = hp_heat
        q_aux_delivered[t] = aux_heat
        kwh_hp[t] = hp_elec_kwh
        therms_aux[t] = gas_therms

        # State update
        net_heat_gain = hp_heat + aux_heat + q_int - heat_loss
        t_in[t + 1] = t_in[t] + (net_heat_gain / c_btu) * dt_hr

    hvac_action[-1] = hvac_action[-2]

    # 5. Format Output Dataframe
    df_res = df_hourly.copy()
    df_res['t_in_f'] = t_in
    df_res['hvac_action'] = hvac_action
    df_res['q_hp_btu'] = q_hp_delivered
    df_res['q_aux_btu'] = q_aux_delivered
    df_res['kwh_hp'] = kwh_hp
    df_res['therms_aux'] = therms_aux

    df_res['elec_cost_usd'] = df_res['kwh_hp'] * elec_rate_per_kwh
    df_res['gas_cost_usd'] = df_res['therms_aux'] * gas_rate_per_therm
    df_res['total_cost_usd'] = (
        df_res['elec_cost_usd'] + df_res['gas_cost_usd']
    )

    df_res['co2_elec_kg'] = df_res['kwh_hp'] * grid_co2_kg_per_kwh
    df_res['co2_gas_kg'] = df_res['therms_aux'] * gas_co2_kg_per_therm
    df_res['total_co2_kg'] = df_res['co2_elec_kg'] + df_res['co2_gas_kg']

    summary = {
        'total_elec_kwh': df_res['kwh_hp'].sum(),
        'total_gas_therms': df_res['therms_aux'].sum(),
        'total_cost_usd': df_res['total_cost_usd'].sum(),
        'total_co2_kg': df_res['total_co2_kg'].sum(),
        'hp_coverage_pct': (
            df_res['q_hp_btu'].sum()
            / max(1.0, (df_res['q_hp_btu'].sum() + df_res['q_aux_btu'].sum()))
        )
        * 100,
        'min_indoor_temp_f': df_res['t_in_f'].min(),
    }

    return df_res, summary

In [ ]:
SIM_SOURCE = 'import numpy as np\nimport pandas as pd\nfrom scipy.interpolate import interp1d\n\n\ndef run_thermal_hp_sim(\n    df_hourly: pd.DataFrame,\n    area_sqft: float = 3310.0,\n    r_val: float = 5.065,\n    setpoint_f: float = 68.0,\n    deadband_f: float = 1.0,\n    # Standard multi-point COP ratings dict: {temp_f: COP}\n    cop_ratings: dict[float, float] | None = None,\n    # Standard multi-point capacity ratings dict: {temp_f: max_btu_per_hr}\n    capacity_ratings: dict[float, float] | None = None,\n    hp_capacity_tons: float = 3.0,  # Used as fallback baseline if capacity_ratings is None\n    gas_furnace_eff: float = 0.80,\n    elec_rate_per_kwh: float = 0.15,\n    gas_rate_per_therm: float = 1.20,\n    grid_co2_kg_per_kwh: float = 0.45,\n    gas_co2_kg_per_therm: float = 5.30,\n) -> tuple[pd.DataFrame, dict]:\n    """Runs an hourly 1-Node RC thermal simulation for a heat pump home with gas backup,\n\n    using 1D linear interpolation for cold-climate COP and capacity curves.\n    """\n    # 1. Provide default NEEP/AHRI benchmark rating curves if none supplied\n    if cop_ratings is None:\n        # Standard cold-climate inverter profile (47°F, 17°F, 5°F, -22°F)\n        cop_ratings = {47.0: 3.60, 17.0: 2.40, 5.0: 1.80, -22.0: 1.00}\n\n    if capacity_ratings is None:\n        # Base 3-ton unit (36,000 BTU/hr nominal at 47°F) with cold weather capacity drop\n        capacity_ratings = {\n            47.0: hp_capacity_tons * 12000.0,\n            17.0: hp_capacity_tons * 12000.0 * 0.78,\n            5.0: hp_capacity_tons * 12000.0 * 0.60,\n            -22.0: hp_capacity_tons * 12000.0 * 0.30,\n        }\n\n    # Sort dictionary keys to ensure ascending arrays for interpolation\n    temps_cop = np.array(sorted(cop_ratings.keys()))\n    vals_cop = np.array([cop_ratings[t] for t in temps_cop])\n\n    temps_cap = np.array(sorted(capacity_ratings.keys()))\n    vals_cap = np.array([capacity_ratings[t] for t in temps_cap])\n\n    # 2. Build SciPy linear interpolation functions\n    get_cop_interp = interp1d(\n        temps_cop,\n        vals_cop,\n        kind=\'linear\',\n        bounds_error=False,\n        fill_value=(vals_cop[0], vals_cop[-1]),  # Clip at boundaries\n    )\n\n    get_cap_interp = interp1d(\n        temps_cap,\n        vals_cap,\n        kind=\'linear\',\n        bounds_error=False,\n        fill_value=(vals_cap[0], vals_cap[-1]),  # Clip at boundaries\n    )\n\n    # 3. Setup Building & Simulation State\n    n_hours = len(df_hourly)\n    ua_btu = area_sqft / r_val\n    c_btu = area_sqft * 15.0\n\n    t_in = np.zeros(n_hours)\n    q_hp_delivered = np.zeros(n_hours)\n    q_aux_delivered = np.zeros(n_hours)\n    kwh_hp = np.zeros(n_hours)\n    therms_aux = np.zeros(n_hours)\n    hvac_action = [\'OFF\'] * n_hours\n\n    t_in[0] = setpoint_f\n    dt_hr = 1.0\n\n    # 4. Hourly Simulation Loop\n    for t in range(n_hours - 1):\n        t_amb = df_hourly[\'t_amb_f\'].iloc[t]\n        q_int = df_hourly[\'q_int_btu\'].iloc[t]\n\n        heat_loss = ua_btu * (t_in[t] - t_amb)\n        heat_needed = max(0.0, heat_loss - q_int)\n\n        if t_in[t] < (setpoint_f - deadband_f) or heat_needed > 0:\n            # Look up interpolated COP and max capacity for current outdoor temp\n            cop_current = float(get_cop_interp(t_amb))\n            avail_hp_cap = float(get_cap_interp(t_amb))\n\n            # Heat delivered split\n            hp_heat = min(heat_needed, avail_hp_cap)\n            aux_heat = max(0.0, heat_needed - hp_heat)\n\n            # Energy inputs\n            hp_elec_kwh = (hp_heat / max(1.0, cop_current)) / 3412.14\n            gas_therms = (aux_heat / gas_furnace_eff) / 100000.0\n\n            if hp_heat > 0 and aux_heat > 0:\n                hvac_action[t] = \'HP_AND_GAS_AUX\'\n            elif hp_heat > 0:\n                hvac_action[t] = \'HP_ONLY\'\n            elif aux_heat > 0:\n                hvac_action[t] = \'GAS_AUX_ONLY\'\n        else:\n            hp_heat = 0.0\n            aux_heat = 0.0\n            hp_elec_kwh = 0.0\n            gas_therms = 0.0\n            hvac_action[t] = \'OFF\'\n\n        q_hp_delivered[t] = hp_heat\n        q_aux_delivered[t] = aux_heat\n        kwh_hp[t] = hp_elec_kwh\n        therms_aux[t] = gas_therms\n\n        # State update\n        net_heat_gain = hp_heat + aux_heat + q_int - heat_loss\n        t_in[t + 1] = t_in[t] + (net_heat_gain / c_btu) * dt_hr\n\n    hvac_action[-1] = hvac_action[-2]\n\n    # 5. Format Output Dataframe\n    df_res = df_hourly.copy()\n    df_res[\'t_in_f\'] = t_in\n    df_res[\'hvac_action\'] = hvac_action\n    df_res[\'q_hp_btu\'] = q_hp_delivered\n    df_res[\'q_aux_btu\'] = q_aux_delivered\n    df_res[\'kwh_hp\'] = kwh_hp\n    df_res[\'therms_aux\'] = therms_aux\n\n    df_res[\'elec_cost_usd\'] = df_res[\'kwh_hp\'] * elec_rate_per_kwh\n    df_res[\'gas_cost_usd\'] = df_res[\'therms_aux\'] * gas_rate_per_therm\n    df_res[\'total_cost_usd\'] = (\n        df_res[\'elec_cost_usd\'] + df_res[\'gas_cost_usd\']\n    )\n\n    df_res[\'co2_elec_kg\'] = df_res[\'kwh_hp\'] * grid_co2_kg_per_kwh\n    df_res[\'co2_gas_kg\'] = df_res[\'therms_aux\'] * gas_co2_kg_per_therm\n    df_res[\'total_co2_kg\'] = df_res[\'co2_elec_kg\'] + df_res[\'co2_gas_kg\']\n\n    summary = {\n        \'total_elec_kwh\': df_res[\'kwh_hp\'].sum(),\n        \'total_gas_therms\': df_res[\'therms_aux\'].sum(),\n        \'total_cost_usd\': df_res[\'total_cost_usd\'].sum(),\n        \'total_co2_kg\': df_res[\'total_co2_kg\'].sum(),\n        \'hp_coverage_pct\': (\n            df_res[\'q_hp_btu\'].sum()\n            / max(1.0, (df_res[\'q_hp_btu\'].sum() + df_res[\'q_aux_btu\'].sum()))\n        )\n        * 100,\n        \'min_indoor_temp_f\': df_res[\'t_in_f\'].min(),\n    }\n\n    return df_res, summary\n'

In [ ]:
_old = "t_in[t + 1] = t_in[t] + (net_heat_gain / c_btu) * dt_hr"
assert _old in SIM_SOURCE
exec(SIM_SOURCE.replace(_old, "t_in[t + 1] = min(setpoint_f, t_in[t] + (net_heat_gain / c_btu) * dt_hr)  # PATCH: cap at setpoint (the original has no cooling, so indoor temp drifts to ~90 F)")
                .replace("def run_thermal_hp_sim(", "def run_thermal_hp_sim_capped("))
df_in = pd.DataFrame({"t_amb_f": w["t_f"].values, "q_int_btu": Q_INT_BTUH}, index=pd.date_range("2026-01-01", periods=n, freq="h"))
def patrick(size_kbtuh):
    caps = {t: size_kbtuh * 1000 * f for t, f in CAP_FRAC.items()}
    _, s = run_thermal_hp_sim_capped(df_in, area_sqft=AREA_SQFT, r_val=R_VAL, setpoint_f=SETPOINT_F, cop_ratings=COP_PTS, capacity_ratings=caps,
        hp_capacity_tons=size_kbtuh / 12, gas_furnace_eff=FURNACE_AFUE, elec_rate_per_kwh=ELEC_USD_KWH, gas_rate_per_therm=GAS_USD_KWH_FUEL * 29.3071,
        grid_co2_kg_per_kwh=0.4173, gas_co2_kg_per_therm=GAS_KG_KWH_FUEL * 29.3071)
    return s
chk = []
for _, r in res_hf[res_hf.size_kbtuh > 0].drop_duplicates("size_kbtuh").iterrows():
    p = patrick(r.size_kbtuh)
    chk.append(dict(size_kbtuh=r.size_kbtuh, enumerated_elec_kwh=r.annual_elec_kwh, patrick_elec_kwh=SCALE * p["total_elec_kwh"],
                    enumerated_gas_therms=r.annual_gas_kwh_fuel / 29.3071, patrick_gas_therms=SCALE * p["total_gas_therms"]))
pd.DataFrame(chk).round(1)

## Notes
- `res` (Gurobi, free dispatch) vs `res_hf` (Patrick's heat-pump-first rule): the gap is the value of a smart controller.
- The cross-check compares Patrick's capped simulation to the enumerated heat-pump-first run, so it validates the load, COP and capacity setup (not the Gurobi dispatch).
- CO2 in the Patrick simulation uses a flat 0.4173 kg/kWh; the Gurobi and enumeration runs use the hourly MISO 2025 file, so only energy (kWh, therms) is compared.
- Not modeled: time-of-use rates, the CenterPoint rule, AC credit, defrost, cooling, thermal mass in the Gurobi model. Placeholders: prices, rebate, cost equation; grid carbon is average, not marginal.